> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 01 — chromosome A/D GRMs and reusable LOCO cache

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
fit_mask=train.notna().any(axis=1).to_numpy()
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
result=vigor.build_grms(PLINK_PREFIX,store=store,components=('additive','dominance'),by_chromosome=True,sample_ids=train.index.astype(str),fit_mask=fit_mask,block_size=GRM_BLOCK_SIZE,min_maf=GRM_MIN_MAF,compute_dtype=GRM_COMPUTE_DTYPE,accum_dtype=GRM_ACCUM_DTYPE,output_dtype=GRM_OUTPUT_DTYPE,blas_threads=BLAS_THREADS,progress=True,reuse=True,persist_genomewide=True)
result.attrs['scientific_signature']

In [ ]:
Ga=vigor.open_grm(store,component='additive',materialize=True)
Gd=vigor.open_grm(store,component='dominance',materialize=True)
print(Ga.shape,Gd.shape)
# LOCO is reconstructed from chromosome caches; no PLINK reread:
Ka_loco1=vigor.open_grm(store,component='additive',loco='1',materialize=False)
print(Ka_loco1.shape)

In [ ]:
# Optional large-N/operator cache. At ~2,369 rats dense matrices are cheap, but caching eigensystems helps repeated GWAS/null fits.
# vigor.cache_grm_spectral(store,component='additive',rank=min(500,Ga.shape[0]-1))